# Query OpenTelemetry data with DuckDB in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smithclay/duckdb-otlp/blob/main/notebooks/duckdb_otlp_colab.ipynb)

This notebook loads the [`duckdb-otlp`](https://github.com/smithclay/duckdb-otlp) extension straight from its GitHub Pages
extension repository and uses it to query OpenTelemetry (OTLP) **traces**, **logs**, and **metrics** with plain SQL —
no collector, no database server.

The extension is **unsigned** (community-built), so we open the connection with `allow_unsigned_extensions` — the Python
equivalent of launching the DuckDB CLI with `duckdb -unsigned`.

## 1. Install DuckDB 1.5.4

DuckDB extension binaries are tied to an **exact** DuckDB version, and the `otlp` extension is published for
`v1.5.4`. Colab ships an older DuckDB (1.3.x), so the cell below installs 1.5.4 to match. If an older DuckDB was
already imported into the kernel, the cell restarts the runtime once — just **re-run it** when the session comes back.

In [ ]:
# Colab preinstalls an older DuckDB (1.3.x). The otlp extension is published for
# DuckDB v1.5.4, and extension binaries are pinned to an exact DuckDB version, so
# install 1.5.4 to match — any other version makes INSTALL request a build that
# isn't published (the IO Error / 404 you may have hit).
REQUIRED = "1.5.4"

import importlib.metadata as ilm, subprocess, sys, os

try:
    installed = ilm.version("duckdb")
except ilm.PackageNotFoundError:
    installed = None
if installed != REQUIRED:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"duckdb=={REQUIRED}"], check=True)

# A restart is only needed if an OLD duckdb was already imported into this kernel.
if "duckdb" in sys.modules and sys.modules["duckdb"].__version__ != REQUIRED:
    print(f"Installed duckdb {REQUIRED}; restarting runtime — re-run this cell when it returns.")
    os.kill(os.getpid(), 9)

import duckdb
print("duckdb", duckdb.__version__)

## 2. Load the `otlp` extension from GitHub Pages

`smithclay.github.io/duckdb-otlp` hosts an unsigned DuckDB extension repository. We point `INSTALL ... FROM` at it and
load it into a connection that allows unsigned extensions.

In [ ]:
PAGES = "https://smithclay.github.io/duckdb-otlp"

# config={'allow_unsigned_extensions': 'true'} == launching the CLI as `duckdb -unsigned`
con = duckdb.connect(config={"allow_unsigned_extensions": "true"})
con.execute(f"INSTALL otlp FROM '{PAGES}'; LOAD otlp;")
print("otlp extension loaded")

## 3. Grab some sample OTLP data

The same site ships small OTLP/JSON samples. The `read_otlp_*` table functions read local files, so download them first.
(They also read OTLP **protobuf** — swap in the `.pb` files if you prefer.)

In [ ]:
import urllib.request

SAMPLES = f"{PAGES}/wasm-demo/samples"
for name in ["traces_simple.jsonl", "logs_simple.jsonl", "metrics_simple.jsonl"]:
    urllib.request.urlretrieve(f"{SAMPLES}/{name}", name)
    print("downloaded", name)

## 4. Traces — slowest spans

`read_otlp_traces` flattens spans into 24 `snake_case` columns. Duration is exposed in nanoseconds as
`duration_time_unix_nano`; divide by `1e6` for milliseconds.

In [ ]:
con.sql("""
    SELECT service_name, name AS span_name, kind,
           duration_time_unix_nano / 1e6 AS duration_ms
    FROM read_otlp_traces('traces_simple.jsonl')
    ORDER BY duration_ms DESC
    LIMIT 10
""")

## 5. Logs — count by severity

`read_otlp_logs` gives 18 columns including `severity_text`, `body`, and trace-correlation fields (`trace_id`, `span_id`).

In [ ]:
con.sql("""
    SELECT service_name, severity_text, count(*) AS n
    FROM read_otlp_logs('logs_simple.jsonl')
    GROUP BY ALL
    ORDER BY n DESC
""")

## 6. Metrics — gauge values

Metrics are split by type. `read_otlp_metrics_gauge` carries `int_value` / `double_value`; sums, histograms, and
exponential histograms have their own `read_otlp_metrics_*` functions.

In [ ]:
con.sql("""
    SELECT service_name, name AS metric_name, unit,
           coalesce(double_value, int_value) AS value
    FROM read_otlp_metrics_gauge('metrics_simple.jsonl')
    LIMIT 10
""")

## Where to go next

- Point the `read_otlp_*` functions at your own exported OTLP `.json` / `.jsonl` / `.pb` files.
- Other functions: `read_otlp_metrics_sum`, `read_otlp_metrics_histogram`, `read_otlp_metrics_exp_histogram`.
- Docs & schema reference: https://smithclay.github.io/duckdb-otlp
- Source: https://github.com/smithclay/duckdb-otlp